# Combine Tracking Videos into Vertical Stack

This notebook combines multiple tracking videos from different tracker configurations (SORT, OC-SORT, ByteTrack) into a single vertically-stacked video for easy comparison.

**Key Features:**
- Videos are stacked vertically (one above the other)
- Supports SORT, OC-SORT, and ByteTrack trackers (in that order)
- Each video is cropped to show only the **top portion** of the frame (configurable percentage)
- This creates a wider aspect ratio for each video cell, focusing on the upper part of the road

In [22]:
import cv2 as cv
import numpy as np
from pathlib import Path
import os
import math
from tqdm import tqdm

## 1. Find All Tracking Videos

In [23]:
# Configuration
RESULTS_DIR = Path("../results")

# Define patterns for each tracker (in order: SORT, OC-SORT, ByteTrack)
TRACKER_PATTERNS = [
    "tracking_sort_finetuned",
    "tracking_ocsort_finetuned",
    "tracking_bytetrack_finetuned"
]

VIDEO_FILENAME = "tracking.avi"
OUTPUT_PATH = Path("../results/tracking_comparison_all_trackers.avi")

# Configurable crop percentage (0.0 to 1.0)
# 0.30 = top 30%, 0.50 = top 50%, 1.0 = full frame
CROP_PERCENTAGE = 0.3

print(f"Crop setting: Top {int(CROP_PERCENTAGE * 100)}% of frame\n")

# Find all directories for each tracker pattern
video_paths = []

for pattern in TRACKER_PATTERNS:
    print(f"Searching for '{pattern}' directories...")
    tracking_dirs = sorted([d for d in RESULTS_DIR.iterdir() 
                           if d.is_dir() and d.name.startswith(pattern)])
    
    for d in tracking_dirs:
        video_path = d / VIDEO_FILENAME
        if video_path.exists():
            video_paths.append((d.name, video_path))
            print(f"  ✓ {d.name}")
        else:
            print(f"  ✗ {d.name} (video not found)")
    print()

print(f"Total videos to combine: {len(video_paths)}")

Crop setting: Top 30% of frame

Searching for 'tracking_sort_finetuned' directories...
  ✓ tracking_sort_finetuned

Searching for 'tracking_ocsort_finetuned' directories...
  ✓ tracking_ocsort_finetuned

Searching for 'tracking_bytetrack_finetuned' directories...
  ✓ tracking_bytetrack_finetuned

Total videos to combine: 3


## 2. Calculate Grid Layout

In [24]:
def calculate_grid_layout(num_videos):
    """
    Calculate grid layout: always vertical stack (N rows, 1 column).
    Returns (rows, cols)
    """
    if num_videos == 0:
        return (0, 0)
    else:
        # Always stack vertically: N rows, 1 column
        return (num_videos, 1)

num_videos = len(video_paths)
rows, cols = calculate_grid_layout(num_videos)

print(f"Grid layout: {rows} rows × {cols} column (vertical stack)")
print(f"Total cells: {rows * cols} (using {num_videos})")
print(f"Note: Videos will be cropped to show only top {int(CROP_PERCENTAGE * 100)}% of frame")

Grid layout: 3 rows × 1 column (vertical stack)
Total cells: 3 (using 3)
Note: Videos will be cropped to show only top 30% of frame


## 3. Load Video Properties

In [25]:
# Open first video to get properties
if len(video_paths) == 0:
    raise ValueError("No videos found to combine!")

first_video = cv.VideoCapture(str(video_paths[0][1]))
original_width = int(first_video.get(cv.CAP_PROP_FRAME_WIDTH))
original_height = int(first_video.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = first_video.get(cv.CAP_PROP_FPS)
total_frames = int(first_video.get(cv.CAP_PROP_FRAME_COUNT))
first_video.release()

# Apply configurable crop to height
crop_height = int(original_height * CROP_PERCENTAGE)

print(f"\nOriginal video properties:")
print(f"  Resolution: {original_width}x{original_height}")
print(f"  Cropped to top {int(CROP_PERCENTAGE * 100)}%: {original_width}x{crop_height}")
print(f"  FPS: {fps}")
print(f"  Total frames: {total_frames}")

# Calculate cell size (resize videos to fit in grid)
# Keep aspect ratio and fit within reasonable output size
MAX_OUTPUT_WIDTH = 1920
MAX_OUTPUT_HEIGHT = 1080

cell_width = MAX_OUTPUT_WIDTH // cols
cell_height = MAX_OUTPUT_HEIGHT // rows

# Maintain aspect ratio of CROPPED frame
aspect_ratio = original_width / crop_height
if cell_width / cell_height > aspect_ratio:
    cell_width = int(cell_height * aspect_ratio)
else:
    cell_height = int(cell_width / aspect_ratio)

output_width = cell_width * cols
output_height = cell_height * rows

print(f"\nOutput video properties:")
print(f"  Cell size: {cell_width}x{cell_height}")
print(f"  Grid resolution: {output_width}x{output_height}")
print(f"  FPS: {fps}")


Original video properties:
  Resolution: 1920x1080
  Cropped to top 30%: 1920x324
  FPS: 10.0
  Total frames: 1606

Output video properties:
  Cell size: 1920x324
  Grid resolution: 1920x972
  FPS: 10.0


## 4. Create Combined Video

In [26]:
# Open all video captures
captures = []
labels = []

for name, path in video_paths:
    cap = cv.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"Warning: Could not open {path}")
        continue
    captures.append(cap)
    
    # Extract tracker type and configuration from folder name
    # Example: tracking_sort_finetuned_age=2_threshold=0.05
    # IMPORTANT: Check ocsort BEFORE sort to avoid substring match
    if "ocsort_finetuned" in name:
        tracker_type = "OC-SORT"
        config_part = name.replace("tracking_ocsort_finetuned_", "").replace("tracking_ocsort_finetuned", "")
    elif "bytetrack_finetuned" in name:
        tracker_type = "ByteTrack"
        config_part = name.replace("tracking_bytetrack_finetuned_", "").replace("tracking_bytetrack_finetuned", "")
    elif "sort_finetuned" in name:
        tracker_type = "SORT"
        config_part = name.replace("tracking_sort_finetuned_", "").replace("tracking_sort_finetuned", "")
    else:
        tracker_type = "Unknown"
        config_part = name
    
    # Extract configuration parameters
    if config_part:
        parts = config_part.split("_")
        label_parts = []
        for part in parts:
            if "=" in part:
                key, val = part.split("=")
                label_parts.append(f"{key}={val}")
        config_str = ", ".join(label_parts) if label_parts else ""
        label = f"{tracker_type}: {config_str}" if config_str else tracker_type
    else:
        label = tracker_type
    
    labels.append(label)

print(f"\nOpened {len(captures)} video captures")
for i, label in enumerate(labels, 1):
    print(f"  {i}. {label}")


Opened 3 video captures
  1. SORT
  2. OC-SORT
  3. ByteTrack


In [27]:
# Create video writer
fourcc = cv.VideoWriter_fourcc(*'XVID')
out = cv.VideoWriter(str(OUTPUT_PATH), fourcc, fps, (output_width, output_height))

if not out.isOpened():
    raise ValueError(f"Could not create output video at {OUTPUT_PATH}")

print(f"\nCreating combined video at {OUTPUT_PATH}...")
print(f"Processing {total_frames} frames...")

# Process frame by frame
for frame_idx in tqdm(range(total_frames), desc="Combining videos", unit="frame"):
    # Create blank canvas
    canvas = np.zeros((output_height, output_width, 3), dtype=np.uint8)
    
    # Read and place each video frame
    for idx, (cap, label) in enumerate(zip(captures, labels)):
        ret, frame = cap.read()
        
        if not ret:
            # If video ended, create a black frame
            frame = np.zeros((original_height, original_width, 3), dtype=np.uint8)
        
        # Crop to top percentage of frame (configurable)
        crop_height = int(original_height * CROP_PERCENTAGE)
        frame_cropped = frame[0:crop_height, :]  # Take only top portion
        
        # Resize cropped frame to cell size
        frame_resized = cv.resize(frame_cropped, (cell_width, cell_height))
        
        # Add label to frame
        label_text = label
        font = cv.FONT_HERSHEY_SIMPLEX
        font_scale = 0.6
        font_thickness = 2
        text_size = cv.getTextSize(label_text, font, font_scale, font_thickness)[0]
        
        # Black background for text (larger padding)
        text_x = 10
        text_y = 30
        padding = 5
        cv.rectangle(frame_resized, 
                    (text_x - padding, text_y - text_size[1] - padding),
                    (text_x + text_size[0] + padding, text_y + padding),
                    (0, 0, 0), -1)
        
        # White text
        cv.putText(frame_resized, label_text, (text_x, text_y),
                  font, font_scale, (255, 255, 255), font_thickness)
        
        # Calculate position in grid
        row = idx // cols
        col = idx % cols
        
        # Place frame in canvas
        y_start = row * cell_height
        y_end = y_start + cell_height
        x_start = col * cell_width
        x_end = x_start + cell_width
        
        canvas[y_start:y_end, x_start:x_end] = frame_resized
    
    # Write frame to output
    out.write(canvas)

# Release resources
for cap in captures:
    cap.release()
out.release()

print(f"\n✅ Combined video saved to: {OUTPUT_PATH}")
print(f"   Resolution: {output_width}x{output_height}")
print(f"   Duration: {total_frames} frames @ {fps} FPS = {total_frames/fps:.2f} seconds")
print(f"   Layout: {rows} videos stacked vertically (top {int(CROP_PERCENTAGE * 100)}% of each video)")
print(f"   Trackers: SORT, OC-SORT, ByteTrack")


Creating combined video at ../results/tracking_comparison_all_trackers.avi...
Processing 1606 frames...


Combining videos:   0%|          | 0/1606 [00:00<?, ?frame/s]

Combining videos: 100%|██████████| 1606/1606 [00:35<00:00, 45.26frame/s]


✅ Combined video saved to: ../results/tracking_comparison_all_trackers.avi
   Resolution: 1920x972
   Duration: 1606 frames @ 10.0 FPS = 160.60 seconds
   Layout: 3 videos stacked vertically (top 30% of each video)
   Trackers: SORT, OC-SORT, ByteTrack
